# N-Power Brief 4 — Chart Rendering Notebook

**An Independent M&E Review of Nigeria's N-Power Programme, 2016–2026: Origin, Performance, and Reform Options**

Author: Usman Almuarif Mashood · Independent Policy Researcher

This notebook renders the three charts used in Brief 4 from the source CSVs in `data/`. Each chart is a separate cell — run any cell independently to regenerate that chart. All PNGs are saved to `charts/` at 300 DPI.

**Data sources:** All CSVs are populated from the 30-source structured evidence matrix maintained in the project repository. Every figure traces to a named public source; contradictions in the record are named, not resolved.

**Usage in Jupyter or Colab:**
1. Ensure `pandas` and `matplotlib` are installed
2. Confirm folder structure: this notebook sits alongside a `data/` folder containing the three CSVs
3. Run each cell in order: Setup → Chart 1 → Chart 2 → Chart 3

**Reproducibility:** re-running this notebook produces byte-identical PNGs against the same input CSVs.


## Setup — imports, house style, folder scaffolding

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import pandas as pd

# ---------------------------------------------------------------------------
# House style — navy + gold, matches Briefs 2 and 3
# ---------------------------------------------------------------------------
NAVY = "#0a2342"
GOLD = "#c9a227"
RED = "#a83232"
GREY_LIGHT = "#e5e5e5"
GREY_MID = "#8a8a8a"
GREY_DARK = "#333333"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.titlecolor": NAVY,
    "axes.labelcolor": GREY_DARK,
    "axes.edgecolor": GREY_MID,
    "axes.linewidth": 0.8,
    "xtick.color": GREY_DARK,
    "ytick.color": GREY_DARK,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Paths
BASE = Path(".").resolve()
DATA = BASE / "data"
CHARTS = BASE / "charts"
CHARTS.mkdir(exist_ok=True, parents=True)

print(f"Base folder: {BASE}")
print(f"Data folder: {DATA} — exists: {DATA.exists()}")
print(f"Charts folder: {CHARTS} — exists: {CHARTS.exists()}")


## Chart 1 — Budget transparency

*What can and cannot be known about N-Power spending, 2016–2024.*

The empty bars are the finding: four of nine years have no publicly disclosed N-Power fiscal figure at all; four are ministry-envelope figures where N-Power's share is undisclosed; only one year (2016) has a verified N-Power-specific figure.


In [ ]:
df = pd.read_csv(DATA / "npower_budget_transparency.csv")

fig, ax = plt.subplots(figsize=(11, 6.5))

x = df["year"].astype(str).tolist()
y = df["value_naira_bn"].tolist()
cats = df["category"].tolist()

color_map = {
    "N-Power verified": NAVY,
    "Ministry envelope": GOLD,
    "Not disclosed": GREY_LIGHT,
}

for i, (yr, val, cat) in enumerate(zip(x, y, cats)):
    colour = color_map[cat]
    if pd.isna(val):
        ax.bar(yr, 20, color=GREY_LIGHT, edgecolor=GREY_MID,
               linewidth=1.0, linestyle="--", hatch="////", alpha=0.6)
        ax.text(i, 26, "Not disclosed", ha="center", va="bottom",
                fontsize=8.5, color=GREY_DARK, style="italic")
    else:
        ax.bar(yr, val, color=colour, edgecolor="grey", linewidth=0.5)
        ax.text(i, val + 12, f"₦{val:,.1f}bn", ha="center", va="bottom",
                fontsize=10, fontweight="bold", color=GREY_DARK)

fig.suptitle("What can and cannot be known about N-Power spending, 2016–2024",
             fontsize=15, fontweight="bold", color=NAVY, x=0.02, ha="left", y=0.97)
plt.figtext(0.02, 0.925,
            "Only one N-Power-specific figure exists in the public record. "
            "All others are ministry envelopes or missing entirely.",
            fontsize=10.5, color=GREY_DARK, style="italic")

ax.set_ylabel("₦ billion")
ax.set_ylim(0, 600)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle=":", color=GREY_MID, alpha=0.5)

legend_elements = [
    mpatches.Patch(facecolor=NAVY, label="N-Power verified"),
    mpatches.Patch(facecolor=GOLD, label="Ministry envelope (share undisclosed)"),
    mpatches.Patch(facecolor=GREY_LIGHT, edgecolor=GREY_MID, hatch="////",
                   label="Not disclosed / no data"),
]
ax.legend(handles=legend_elements, loc="upper center",
          bbox_to_anchor=(0.5, 1.02), frameon=False,
          fontsize=10, ncol=3)

plt.figtext(0.02, 0.02,
            "Sources: TheCable (2019, 2021) for 2016; BudgIT tracker (2020–2023) "
            "for ministry-level FMHDMSD envelopes. N-Power share of ministry envelopes is not publicly disclosed. "
            "2017–2019 and 2024 have no disclosed N-Power figure. The empty bars are the finding.",
            fontsize=8, color=GREY_MID, style="italic", wrap=True)

plt.subplots_adjust(top=0.82, bottom=0.15)
out = CHARTS / "01_budget_transparency.png"
plt.savefig(out)
plt.show()
print(f"Saved: {out}")


## Chart 2 — Reach verification

*Beneficiary claims vs. independent verification across N-Power batches.*

Large confident claims against thin, fragmentary verification. The asymmetry, not any single number, is the finding. The ghost-name uncertainty band (9–18% of claimed reach) is shown as red tick marks bounding the verified navy range. N-Teach and N-Health cumulative claims have no independent verification stream at all.


In [ ]:
df = pd.read_csv(DATA / "npower_reach_verification.csv")
df = df.iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 7))
labels = df["batch_strand"].tolist()

for i, row in df.iterrows():
    claim = row["official_claim"]
    v_low = row["verified_low"]
    v_high = row["verified_high"]
    ghost = row["ghost_flag"] == "Y"

    ax.barh(i, claim, color=GOLD, alpha=0.45,
            edgecolor=GOLD, linewidth=0.6, height=0.65)

    if v_low > 0 and v_high > 0:
        ax.barh(i, v_high - v_low, left=v_low, color=NAVY, alpha=0.90,
                edgecolor=NAVY, linewidth=0.5, height=0.65)
        if ghost:
            ax.plot([v_low, v_low], [i - 0.30, i + 0.30],
                    color=RED, linewidth=2.5, zorder=5)
            ax.plot([v_high, v_high], [i - 0.30, i + 0.30],
                    color=RED, linewidth=2.5, zorder=5)

    ax.text(claim + 20000, i, f"claim: {int(claim):,}",
            va="center", fontsize=10, color=GREY_DARK, fontweight="bold")

    if v_low > 0 and v_high > 0:
        ax.text(v_high - 5000, i - 0.42,
                f"verified range: {int(v_low):,} – {int(v_high):,}",
                va="center", ha="right", fontsize=8.5, color=NAVY, style="italic")
    else:
        ax.text(claim / 2, i - 0.42, "no independent verification stream",
                va="center", ha="center", fontsize=8.5, color=RED,
                style="italic", fontweight="bold")

fig.suptitle("Beneficiary claims vs. independent verification, N-Power batches",
             fontsize=15, fontweight="bold", color=NAVY, x=0.02, ha="left", y=0.97)
plt.figtext(0.02, 0.925,
            "Large confident claims against thin, fragmentary verification. "
            "The asymmetry — not any single number — is the finding.",
            fontsize=10.5, color=GREY_DARK, style="italic")

ax.set_yticks(list(range(len(df))))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Beneficiaries")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_xlim(0, 1_150_000)
ax.set_axisbelow(True)
ax.grid(axis="x", linestyle=":", color=GREY_MID, alpha=0.5)

ghost_text = ("Ghost-name adjustment (~9–18% of claimed reach): "
              "70,000 documented over four years (TheCable 2019/2021) plus "
              "20,000 (\"D'Banj list\", 2021, partially overlapping). "
              "The two lists are not clearly additive. Verified-low reflects the 9% adjustment; "
              "verified-high the 18%. Red tick marks bound the ghost-name uncertainty.")
ax.text(0.5, -0.22, ghost_text, transform=ax.transAxes,
        fontsize=8.5, color=GREY_DARK, ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f7f5ec",
                  edgecolor=GOLD, linewidth=0.8), style="italic", wrap=True)

legend_elements = [
    mpatches.Patch(facecolor=GOLD, alpha=0.45, label="Official claim"),
    mpatches.Patch(facecolor=NAVY, alpha=0.90, label="Independently verified range"),
    mpatches.Patch(facecolor="none", edgecolor=RED, linewidth=2,
                   label="Ghost-name uncertainty (9–18% band)"),
]
ax.legend(handles=legend_elements, loc="upper center",
          bbox_to_anchor=(0.5, 1.03), frameon=False, fontsize=9.5, ncol=3)

plt.figtext(0.02, 0.02,
            "Sources: N-Power Information Guide (2017); NSIPA (2026); Premium Times (2019); "
            "Punch (2020); TheCable (2019, 2021); Sahara Reporters (2022); FIJ (2024); ANEEJ (2025); "
            "Osimen et al. (2025); Guardian Nigeria (2024). N-Teach and N-Health cumulative claims "
            "have no independent verification stream.",
            fontsize=8, color=GREY_MID, style="italic", wrap=True)

plt.subplots_adjust(top=0.82, bottom=0.28, left=0.16, right=0.96)
out = CHARTS / "02_reach_verification.png"
plt.savefig(out)
plt.show()
print(f"Saved: {out}")


## Chart 3 — Stipend erosion

*Real-terms value of the N-Power monthly stipend, NBS Food CPI deflated, 2016–2026.*

The nominal ₦30,000 stipend was set at launch and held constant through 2026. Deflated by NBS Food CPI, the 2016 stipend retains an estimated ₦3,000–₦5,000 of its original purchasing power in 2026 — roughly one-sixth to one-tenth. Restoring 2016 real value would require a 2026 nominal stipend of ₦180,000–₦300,000, six to ten times higher than the current rate. No indexation mechanism has been gazetted at any point in the programme's ten-year history.


In [ ]:
df = pd.read_csv(DATA / "npower_stipend_real_value.csv")

fig, ax = plt.subplots(figsize=(11, 7))

years = df["year"].tolist()
nominal = df["nominal_stipend"].tolist()
real_low = df["real_value_low"].tolist()
real_high = df["real_value_high"].tolist()
real_mid = [(l + h) / 2 for l, h in zip(real_low, real_high)]

ax.plot(years, nominal, color=GREY_MID, linestyle="--", linewidth=2,
        label="Nominal stipend (₦30,000 held flat)", zorder=2)

ax.fill_between(years, real_low, real_high, color=NAVY, alpha=0.20,
                label="Real value band (2016 ₦, NBS Food CPI deflated)")
ax.plot(years, real_mid, color=NAVY, linewidth=2.8, marker="o",
        markersize=7, zorder=3)

ax.annotate("₦30,000\n(2016 launch)",
            xy=(2016, 30000), xytext=(2016.3, 45000),
            fontsize=10, fontweight="bold", color=NAVY,
            arrowprops=dict(arrowstyle="->", color=NAVY, lw=1.2))

ax.annotate(f"₦{real_low[-1]:,.0f}–₦{real_high[-1]:,.0f}\n(2026, real value)",
            xy=(2026, real_mid[-1]), xytext=(2024.3, 18000),
            fontsize=10, fontweight="bold", color=RED,
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.2))

ax.annotate("Silent erosion:\nno indexation mechanism\nat any point 2016–2026",
            xy=(2021, real_mid[5]), xytext=(2019, 42000),
            fontsize=9, color=GREY_DARK, style="italic",
            arrowprops=dict(arrowstyle="->", color=GREY_MID, lw=0.8))

fig.suptitle("₦30,000 in 2016 buys what ₦3,000–₦5,000 buys in 2026",
             fontsize=15, fontweight="bold", color=NAVY, x=0.02, ha="left", y=0.97)
plt.figtext(0.02, 0.925,
            "Real-terms value of the N-Power monthly stipend, NBS Food CPI deflated, 2016–2026",
            fontsize=10.5, color=GREY_DARK, style="italic")

ax.set_ylabel("₦ (2016 real value, monthly stipend)")
ax.set_xlabel("Year")
ax.set_ylim(0, 55000)
ax.set_xticks(years)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"₦{int(v):,}"))
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle=":", color=GREY_MID, alpha=0.4)

ax.legend(loc="upper right", frameon=False, fontsize=10)

restoration_text = (
    "Implied 2026 restoration stipend: ₦180,000–₦300,000. "
    "To restore the 2016 real value of ₦30,000, the 2026 nominal stipend would need to be six to ten times higher. "
    "The Renewed Hope reset has not published an indexation rule."
)
ax.text(0.5, -0.20, restoration_text, transform=ax.transAxes,
        fontsize=9.5, color="#8a6d0a", ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff8e6",
                  edgecolor=GOLD, linewidth=0.8), style="italic")

plt.figtext(0.02, 0.02,
            "Sources: N-Power Information Guide (2017) for nominal rate; NBS Consumer Price Index "
            "Food sub-index (annual averages) for deflator. Nominal rate held constant 2016–2024; no indexation "
            "mechanism gazetted at any point. 2026 band reflects re-basing sensitivity — NBS revised its CPI base year in 2024. "
            "Food-CPI deflator used, consistent with Brief 3.",
            fontsize=8, color=GREY_MID, style="italic", wrap=True)

plt.subplots_adjust(top=0.82, bottom=0.24, left=0.10, right=0.95)
out = CHARTS / "03_stipend_erosion.png"
plt.savefig(out)
plt.show()
print(f"Saved: {out}")


---

## Session notes

If any cell fails:
- Verify the three CSVs exist in `data/`: `npower_budget_transparency.csv`, `npower_reach_verification.csv`, `npower_stipend_real_value.csv`
- Verify pandas and matplotlib are installed: `pip install pandas matplotlib`
- Re-run the Setup cell before running any chart cell

If you re-open this notebook in a fresh session, run the Setup cell first — the other cells assume paths and rc-params defined there.

**Next steps after rendering:**
1. Copy the three PNGs from `charts/` into your brief document (Word, LaTeX, or Quarto)
2. Also copy into the LinkedIn dashboard HTML source
3. Commit `data/`, `charts/`, and this notebook to the GitHub repository under `brief_04_NPOWER/analysis/`
